# 01 · Tiếp nhận yêu cầu và quét dữ liệu

Khách hàng điền form 10 mục bên dưới và chọn file dữ liệu. Notebook tạo `runs/<job_id>/form.json`, catalog cột và prompt để AI 1 hoàn thiện yêu cầu. Dữ liệu mẫu không được gửi đi tự động; mặc định không đưa giá trị mẫu vào prompt.

In [1]:
from pathlib import Path
import json
import uuid
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'README.md').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy thư mục gốc repo (cần README.md và src/).')

ROOT = find_repo_root()
JOB_ID = 'job_' + uuid.uuid4().hex[:8]
RUN_DIR = ROOT / 'runs' / JOB_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Sửa đường dẫn này tới file dữ liệu của khách hàng.
DATA_PATH = Path(r'')
TABLE_NAME = None  # Tùy chọn: tên bảng mong muốn; mặc định lấy tên sheet/tệp.
INCLUDE_SAMPLE_VALUES = False  # Bật chỉ khi được phép đưa ví dụ dữ liệu vào prompt.

FORM = {
    'report_name': '',
    'business_goal': '',
    'audience': '',
    'decisions': '',
    'kpis': [],
    'dimensions': [],
    'date_column': '',
    'filters': [],
    'preferred_visuals': [],
    'design_preferences': ''
}

print('Repo:', ROOT)
print('Job:', JOB_ID)
print('Điền FORM và DATA_PATH, sau đó chạy cell tiếp theo.')

Repo: e:\PBI Automation
Job: job_95474c7e
Điền FORM và DATA_PATH, sau đó chạy cell tiếp theo.


In [2]:
def read_tables(path):
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return {path.stem: pd.read_csv(path)}
    if suffix in {'.xlsx', '.xlsm'}:
        book = pd.ExcelFile(path)
        return {sheet: pd.read_excel(path, sheet_name=sheet) for sheet in book.sheet_names}
    if suffix == '.parquet':
        return {path.stem: pd.read_parquet(path)}
    if suffix == '.json':
        frame = pd.read_json(path)
        return {path.stem: frame}
    raise ValueError(f'Định dạng chưa hỗ trợ: {suffix}. Dùng CSV, XLSX, XLSM, Parquet hoặc JSON.')

if not str(DATA_PATH).strip() or not DATA_PATH.is_file():
    raise FileNotFoundError('Hãy đặt DATA_PATH tới một file dữ liệu tồn tại rồi chạy lại cell này.')
if any(value in (None, '') for key, value in FORM.items() if key in {'report_name', 'business_goal'}):
    print('Lưu ý: hãy hoàn thiện các mục bắt buộc của FORM trước khi gửi prompt cho AI.')

tables = read_tables(DATA_PATH)
catalog = {'source_file': str(DATA_PATH.resolve()), 'tables': []}
for table_name, frame in tables.items():
    columns = []
    for column in frame.columns:
        series = frame[column]
        item = {
            'name': str(column),
            'inferred_type': str(series.dtype),
            'nullable': bool(series.isna().any()),
            'null_rate': round(float(series.isna().mean()), 4),
            'unique_count': int(series.nunique(dropna=True))
        }
        if INCLUDE_SAMPLE_VALUES:
            item['sample_values'] = [str(v) for v in series.dropna().head(3).tolist()]
        columns.append(item)
    catalog['tables'].append({'name': TABLE_NAME or table_name, 'row_count': len(frame), 'columns': columns})

(RUN_DIR / 'form.json').write_text(json.dumps(FORM, ensure_ascii=False, indent=2), encoding='utf-8')
(RUN_DIR / 'data_catalog.json').write_text(json.dumps(catalog, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã ghi:', RUN_DIR / 'form.json')
print('Đã ghi:', RUN_DIR / 'data_catalog.json')
display(pd.DataFrame([{**{'table': table['name']}, **column} for table in catalog['tables'] for column in table['columns']]))

FileNotFoundError: Hãy đặt DATA_PATH tới một file dữ liệu tồn tại rồi chạy lại cell này.

## AI 1 · Hoàn thiện yêu cầu
Chạy cell để tạo prompt, gửi prompt cho AI bằng kênh được tổ chức cho phép. AI phải ghi giả định và câu hỏi còn thiếu, không tự khẳng định dữ kiện nghiệp vụ.

In [ ]:
prompt = f'''Bạn là chuyên gia phân tích yêu cầu Power BI. Hoàn thiện form dựa trên form khách hàng và catalog dữ liệu.
Quy tắc: giữ nguyên thông tin đã có; không bịa quy tắc nghiệp vụ; mọi giá trị suy luận phải có lý do; ghi câu hỏi chưa thể kết luận.
Trả về duy nhất JSON hợp lệ theo cấu trúc: {{"requirement": {{...form đã hoàn thiện...}}, "assumptions": [{{"field": "...", "value": "...", "reason": "..."}}], "open_questions": ["..."]}}.

FORM:
{json.dumps(FORM, ensure_ascii=False, indent=2)}

DATA CATALOG:
{json.dumps(catalog, ensure_ascii=False, indent=2)}
'''
print(prompt)

In [ ]:
AI1_RESPONSE = r'''PASTE_AI_JSON_HERE'''

if AI1_RESPONSE.strip() != 'PASTE_AI_JSON_HERE':
    result = json.loads(AI1_RESPONSE)
    if not isinstance(result.get('requirement'), dict):
        raise ValueError('JSON AI 1 phải có object requirement.')
    result['source_job_id'] = JOB_ID
    (RUN_DIR / 'requirement.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Đã ghi:', RUN_DIR / 'requirement.json')
else:
    print('Dán phản hồi JSON của AI vào AI1_RESPONSE để lưu requirement.json.')